In [21]:
import pandas as pd
import os

In [22]:
# 1. Cargar el dataset principal
df_main = pd.read_csv('datos_investigacion_final.csv')

def procesar_guion(imdb_id):
    # Definimos la ruta a la carpeta y el archivo
    folder_name = "CSVs limpios"
    filename = os.path.join(folder_name, f"{imdb_id}.csv")
    
    # Comprobar si el archivo existe
    if not os.path.exists(filename):
        print(f"Aviso: No se encontró el archivo {filename}")
        return None
    
    try:
        # Leer el CSV (ahora que tiene cabeceras correctas)
        df_script = pd.read_csv(filename)
        
        # Limpieza de datos
        # 1. Nombres de personajes a mayúsculas y sin espacios
        df_script['character'] = df_script['character'].astype(str).str.strip().str.upper()
        
        # 2. Diálogos: quitar espacios y limpiar comillas residuales (si las hubiera)
        df_script['dialog'] = (df_script['dialog']
                               .astype(str)
                               .str.strip()
                               .str.replace(r'^"|"$', '', regex=True)) # Quita comillas al inicio/final
        
        # 3. Eliminar filas donde el diálogo o el personaje estén vacíos
        df_script = df_script.dropna(subset=['character', 'dialog'])
        
        # Crear el diccionario: cada personaje es una llave y su valor es TODO su diálogo unido
        # Ejemplo: {'CLAUDIA': 'Hola a todo el mundo. Qué sugieres?', ...}
        guion_dict = df_script.groupby('character')['dialog'].apply(lambda x: ' '.join(x)).to_dict()
        
        return guion_dict

    except Exception as e:
        print(f"Error procesando {imdb_id}: {e}")
        return None

In [23]:
# 2. Crear la nueva columna en el dataset
print("Procesando guiones y creando el corpus...")
df_main['Script_Dict'] = df_main['IMDb_ID'].apply(procesar_guion)

Procesando guiones y creando el corpus...
Aviso: No se encontró el archivo CSVs limpios\tt20215234.csv


In [24]:
# 3. Guardar el resultado
# Guardamos en .pkl para mantener el formato de diccionario intacto
df_main.to_pickle('dataset_analisis_corpus.pkl')

# También guardamos en .csv por si quieres verlo en Excel (aunque los diccionarios se verán como texto)
df_main.to_csv('dataset_analisis_corpus.csv', index=False)

In [25]:
print("¡Proceso completado!")
# Ejemplo de visualización:
# print(df_main[['Title', 'Script_Dict']].head())

¡Proceso completado!
